In [10]:
import pandas as pd
import numpy as np
import hnswlib as hl
import hnsw_benchmark as bm
import twonn
import farhest_point_order as fp
import matplotlib.pyplot as plt
import os

In [11]:
# Alapadatok megadása
dataset_names = [
    "arguana",
    "nfcorpus"
]
embedding_models = [
    "e5-small-v2",
    "multilingual-e5-large",
    "bge-small-en",
    "stella-base-en-v2",
    "all-MiniLM-L6-v2"
]
options = [
    # "_lid_fps_lpca_2nn",
    # "_lid_fps",
    "_lid",
    "_lid",
    "_lid",
    "_lid",
    "_lid"
]

df_corpus = {}
for dataset_name in dataset_names:
    for embedding_model, option in zip(embedding_models, options):
        corpus_filename = dataset_name + "_corpus_" + embedding_model + option

        if dataset_name == 'arguana': # kell, ki kell majd venni
            if embedding_model == 'e5-small-v2':
                corpus_filename = corpus_filename + "_fps_lpca_2nn"
            if embedding_model == 'multilingual-e5-large':
                corpus_filename = corpus_filename + "_fps"

        corpus_input_path = os.path.join("..\\data\\", corpus_filename + ".json")
        corpus_data = pd.read_json(corpus_input_path, orient="records", lines=True)
        df_corpus[dataset_name + "_" + embedding_model] = corpus_data

df_query = {}
for dataset_name in dataset_names:
    for embedding_model in embedding_models:
        query_filename = dataset_name + "_query_" + embedding_model
        query_input_path = os.path.join("..\\data\\", query_filename + ".json")
        query_data = pd.read_json(query_input_path, orient="records", lines=True)
        df_query[dataset_name + "_" + embedding_model] = query_data

In [12]:
for name in df_corpus:
    print("-- " + name + " --")

    print("Corpus data")
    corpus_vectors = np.asarray([np.asarray(v) for v in df_corpus[name].vectors], dtype=np.float32)
    print(corpus_vectors.shape)
    print(df_corpus[name].keys())

    print("Query data")
    query_vectors = np.asarray([np.asarray(v) for v in df_query[name].vectors], dtype=np.float32)
    print(query_vectors.shape)

    print()


-- arguana_e5-small-v2 --
Corpus data
(8674, 384)
Index(['vectors', 'LID', 'fp_score', 'LPCA_dim', 'TwoNN'], dtype='str')
Query data
(1406, 384)

-- arguana_multilingual-e5-large --
Corpus data
(8674, 1024)
Index(['vectors', 'LID', 'fp_score'], dtype='str')
Query data
(1406, 1024)

-- arguana_bge-small-en --
Corpus data
(8674, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 384)

-- arguana_stella-base-en-v2 --
Corpus data
(8674, 768)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 768)

-- arguana_all-MiniLM-L6-v2 --
Corpus data
(8674, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 384)

-- nfcorpus_e5-small-v2 --
Corpus data
(3633, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(3237, 384)

-- nfcorpus_multilingual-e5-large --
Corpus data
(3633, 1024)
Index(['vectors', 'LID'], dtype='str')
Query data
(3237, 1024)

-- nfcorpus_bge-small-en --
Corpus data
(3633, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(3237, 384)

-- nfcorpus

In [61]:
# Benchmarkok futtatása
def run_benchmark(corpus_data, sorted_indices, query_data, n_times=1, ef_search=10):
    recalls = []
    for _ in range(0, n_times):
        bm_data = [corpus_data[i] for i in sorted_indices]
        benchmark = bm.HNSW_benchmark(data=bm_data)
        recall_list = benchmark.run(qs=query_data, ef_search=ef_search)
        recalls.append(np.mean(recall_list))

    print(f"Minimum recall: {np.min(recalls):.4f}")
    print(f"Average recall: {np.mean(recalls):.4f}")
    print(f"Maximum recall: {np.max(recalls):.4f}")

    # plt.figure(figsize=(8, 5))
    # plt.hist(recalls, bins=10, edgecolor="black", alpha=0.75)
    # plt.title("Distribution of recall values")
    # plt.xlabel("Recall")
    # plt.ylabel("Count")
    # plt.grid(axis="y", alpha=0.3)
    # plt.ylim(0, len(df_query.vectors))
    # plt.show()
    return recalls

In [64]:
# HNSW benchmark - alap beállítások
n_times = 1
ef_search = 10

df_bm = pd.DataFrame(columns=[
    "em",
    "dn",
    "random",
    "lid_d",
    "lid_a"
])

# df_bm2 = pd.DataFrame(columns=[
#     "em",
#     "random",
#     "lid_d"
# ])

In [65]:
# HNSW benchmark - Random beszúrás
print("-- Random --")
print()
print()
for em in embedding_models:
    print("-- " + em + " --")
    rcs = []
    for dn in dataset_names:
        print("  -- " + dn + " --")

        sorted_indices = np.arange(len(df_corpus[dn + "_" + em].vectors))
        rc = run_benchmark(df_corpus[dn + "_" + em].vectors, sorted_indices, df_query[dn + "_" + em].vectors, n_times, ef_search)
        df_bm.loc[len(df_bm)] = [em, dn, np.mean(rc), .0, .0]
        rcs.append(rc)

        print()
    print(f"Avg recall: {np.mean(rcs):.4f}")
    print()
    

-- Random --


-- e5-small-v2 --
  -- arguana --
Minimum recall: 0.9146
Average recall: 0.9146
Maximum recall: 0.9146

  -- nfcorpus --
Minimum recall: 0.7028
Average recall: 0.7028
Maximum recall: 0.7028

Avg recall: 0.8087

-- multilingual-e5-large --
  -- arguana --
Minimum recall: 0.9231
Average recall: 0.9231
Maximum recall: 0.9231

  -- nfcorpus --
Minimum recall: 0.7068
Average recall: 0.7068
Maximum recall: 0.7068

Avg recall: 0.8150

-- bge-small-en --
  -- arguana --
Minimum recall: 0.9292
Average recall: 0.9292
Maximum recall: 0.9292

  -- nfcorpus --
Minimum recall: 0.6333
Average recall: 0.6333
Maximum recall: 0.6333

Avg recall: 0.7812

-- stella-base-en-v2 --
  -- arguana --
Minimum recall: 0.9302
Average recall: 0.9302
Maximum recall: 0.9302

  -- nfcorpus --
Minimum recall: 0.7404
Average recall: 0.7404
Maximum recall: 0.7404

Avg recall: 0.8353

-- all-MiniLM-L6-v2 --
  -- arguana --
Minimum recall: 0.9585
Average recall: 0.9585
Maximum recall: 0.9585

  -- nfcorpus -

In [66]:
# HNSW benchmark - LID szerint csökkenő
print("-- Descending by LID --")
print()
print()
for em in embedding_models:
    print("-- " + em + " --")
    rcs = []
    for dn in dataset_names:
        print("  -- " + dn + " --")

        sorted_indices = np.argsort(df_corpus[dn + "_" + em].LID, descending=True)
        rc = run_benchmark(df_corpus[dn + "_" + em].vectors, sorted_indices, df_query[dn + "_" + em].vectors, n_times, ef_search)
        # df_bm.loc[len(df_bm)] = [em, dn, np.mean(rc), 0]
        # df_bm.loc[(df_bm.em == em and df_bm.dn == dn), 'lid_d'] = np.mean(rc)
        upd = df_bm.em == em
        upd &= df_bm.dn == dn
        df_bm.loc[upd, 'lid_d'] = np.mean(rc)
        rcs.append(rc)

        print()
    print(f"Avg recall: {np.mean(rcs):.4f}")
    print()

-- Descending by LID --


-- e5-small-v2 --
  -- arguana --
Minimum recall: 0.9518
Average recall: 0.9518
Maximum recall: 0.9518

  -- nfcorpus --
Minimum recall: 0.7416
Average recall: 0.7416
Maximum recall: 0.7416

Avg recall: 0.8467

-- multilingual-e5-large --
  -- arguana --
Minimum recall: 0.9575
Average recall: 0.9575
Maximum recall: 0.9575

  -- nfcorpus --
Minimum recall: 0.7492
Average recall: 0.7492
Maximum recall: 0.7492

Avg recall: 0.8533

-- bge-small-en --
  -- arguana --
Minimum recall: 0.9650
Average recall: 0.9650
Maximum recall: 0.9650

  -- nfcorpus --
Minimum recall: 0.7572
Average recall: 0.7572
Maximum recall: 0.7572

Avg recall: 0.8611

-- stella-base-en-v2 --
  -- arguana --
Minimum recall: 0.9604
Average recall: 0.9604
Maximum recall: 0.9604

  -- nfcorpus --
Minimum recall: 0.8060
Average recall: 0.8060
Maximum recall: 0.8060

Avg recall: 0.8832

-- all-MiniLM-L6-v2 --
  -- arguana --
Minimum recall: 0.9868
Average recall: 0.9868
Maximum recall: 0.9868

  --

In [67]:
# HNSW benchmark - LID szerint növekvő
print("-- Ascending by LID --")
print()
print()
for em in embedding_models:
    print("-- " + em + " --")
    rcs = []
    for dn in dataset_names:
        print("  -- " + dn + " --")

        sorted_indices = np.argsort(df_corpus[dn + "_" + em].LID, descending=False)
        rc = run_benchmark(df_corpus[dn + "_" + em].vectors, sorted_indices, df_query[dn + "_" + em].vectors, n_times, ef_search)
        upd = df_bm.em == em
        upd &= df_bm.dn == dn
        df_bm.loc[upd, 'lid_a'] = np.mean(rc)
        rcs.append(rc)

        print()
    print(f"Avg recall: {np.mean(rcs):.4f}")
    print()

-- Ascending by LID --


-- e5-small-v2 --
  -- arguana --
Minimum recall: 0.9359
Average recall: 0.9359
Maximum recall: 0.9359

  -- nfcorpus --
Minimum recall: 0.6914
Average recall: 0.6914
Maximum recall: 0.6914

Avg recall: 0.8136

-- multilingual-e5-large --
  -- arguana --
Minimum recall: 0.9341
Average recall: 0.9341
Maximum recall: 0.9341

  -- nfcorpus --
Minimum recall: 0.6938
Average recall: 0.6938
Maximum recall: 0.6938

Avg recall: 0.8139

-- bge-small-en --
  -- arguana --
Minimum recall: 0.9367
Average recall: 0.9367
Maximum recall: 0.9367

  -- nfcorpus --
Minimum recall: 0.6284
Average recall: 0.6284
Maximum recall: 0.6284

Avg recall: 0.7825

-- stella-base-en-v2 --
  -- arguana --
Minimum recall: 0.9413
Average recall: 0.9413
Maximum recall: 0.9413

  -- nfcorpus --
Minimum recall: 0.7251
Average recall: 0.7251
Maximum recall: 0.7251

Avg recall: 0.8332

-- all-MiniLM-L6-v2 --
  -- arguana --
Minimum recall: 0.9798
Average recall: 0.9798
Maximum recall: 0.9798

  -- 

In [68]:
print(df_bm)
# df_bm.loc[df_bm.em == em, 'lid_d'] = 155
# print(df_bm)
# df_bm_avg = df_bm.groupby(['em'])['random', 'lid_d'].mean()
df_bm_avg = df_bm.groupby("em", as_index=False)[["random", "lid_d", "lid_a"]].mean()
print(df_bm_avg)

                      em        dn    random     lid_d     lid_a
0            e5-small-v2   arguana  0.914580  0.951778  0.935917
1            e5-small-v2  nfcorpus  0.702780  0.741551  0.691381
2  multilingual-e5-large   arguana  0.923115  0.957468  0.934068
3  multilingual-e5-large  nfcorpus  0.706827  0.749181  0.693760
4           bge-small-en   arguana  0.929161  0.965007  0.936700
5           bge-small-en  nfcorpus  0.633302  0.757152  0.628390
6      stella-base-en-v2   arguana  0.930156  0.960384  0.941323
7      stella-base-en-v2  nfcorpus  0.740408  0.805962  0.725085
8       all-MiniLM-L6-v2   arguana  0.958535  0.986771  0.979801
9       all-MiniLM-L6-v2  nfcorpus  0.784863  0.824251  0.761940
                      em    random     lid_d     lid_a
0       all-MiniLM-L6-v2  0.871699  0.905511  0.870870
1           bge-small-en  0.781232  0.861079  0.782545
2            e5-small-v2  0.808680  0.846664  0.813649
3  multilingual-e5-large  0.814971  0.853325  0.813914
4      ste

In [69]:
df_filename = "df_bm_v1"
df_output_path = os.path.join("..\\benchmark\\", df_filename + ".csv")
df_bm.to_csv(df_output_path)

df_filename = "df_bm_avg_v1"
df_output_path = os.path.join("..\\benchmark\\", df_filename + ".csv")
df_bm_avg.to_csv(df_output_path)